# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

Entities are referenced by their `@id`. Let's discover available record sets with their fields/columns.

In [ ]:
# List all record sets and their `@id`s, field ids, and columns
print('Record sets available in this dataset:')
for record_set in dataset.record_sets:
    print(f"\nRecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print('  Fields (@id):')
    for field in record_set.fields:
        print(f"    - {field.id} ({field.name})")
        if hasattr(field, 'columns') and field.columns:
            print('      Backed by columns:')
            for col in field.columns:
                print(f"        - {col.id} ({col.name if hasattr(col, 'name') else ''})")

You can print a sample record for each record set below:

In [ ]:
# Print one record from each record set as an example
for record_set in dataset.record_sets:
    print(f"\nSample from RecordSet {record_set.id}:")
    records = dataset.records(record_set=record_set.id)
    for idx, rec in enumerate(records):
        print(rec)
        if idx >= 0:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Reference record sets by their @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# For demonstration, pick the first record set:
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in main DataFrame ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Note:** We'll attempt EDA on the main record set and numeric fields discovered.

In [ ]:
import numpy as np

df = dataframes[main_record_set_id]
# List candidate numeric fields, e.g., columns containing float or int
candidate_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric fields detected:", candidate_numeric_fields)

# If any numeric field is present, pick the first for demo
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    print(f"\nAnalyzing field: {numeric_field}\n")
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}) : {len(filtered_df)} rows")
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized sample:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by first non-numeric field
    candidate_group_fields = [c for c in df.columns if c not in candidate_numeric_fields]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in the record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and candidate_numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if len(candidate_numeric_fields) > 1:
        # Scatter plot for two numeric fields
        plt.figure(figsize=(6, 6))
        sns.scatterplot(data=df, x=candidate_numeric_fields[0], y=candidate_numeric_fields[1])
        plt.title(f"{candidate_numeric_fields[0]} vs {candidate_numeric_fields[1]}")
        plt.xlabel(candidate_numeric_fields[0])
        plt.ylabel(candidate_numeric_fields[1])
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* using `mlcroissant`.

- We programmatically discovered record sets, fields, and their `@id`s.
- Loaded the main record set into a DataFrame.
- Identified numeric fields and performed basic filtering, normalization, and group analysis.
- Visualized field distributions.

**Next steps** could involve domain-specific analysis of the protein data, feature engineering, or more advanced statistical exploration.

\*Note: All entity references use the original Croissant `@id` for robust, schema-driven analysis.*